# Notes
[2026-06-01]:
Basically the other notebook is a suboptimal to run on a computer. [MOST OF THE TRANSFORMER CODE FROM HERE](https://medium.com/data-science/build-your-own-transformer-from-scratch-using-pytorch-84c850470dcb).

Length size of 72 was used because 1. It contains 700+ seq. 2.

> This model is extremely inefficient as ive since learned. I decided to use a encoder-dcoder architecture, since I approached it as a "translation" problem, but after I built it I realsed it only requires new embeddings per residue.

[2026-06-03]: This architecture needs changing! Current encoder-decoder model is inefficient, and could be better solved using an encoder-only architecture. Test Loss: 0.4879, Test Accuracy: 82.17%

[2026-06-04]: After changing the architecture, we're getting *marginally* better results: Test Loss: 0.8457, Test Accuracy: 82.06%
Changing DNN ReLU to GELU did marginally improve results.

Introducing a local convolution layer further improved our results: Test Loss: 0.6680 Test Accuracy: 83.41%


# Loading in the data

In [1]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import numpy as np
from torch import nn
# import torch.nn.functional as F
# import torch.optim as optim


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using: {device}")

from bin.data_loader import load_data

X, y, AminoAcids, SecondaryStructures, aa_stoi, aa_itos, aa_encode, aa_decode, ss_stoi, ss_itos, ss_encode, ss_decode, set_length = load_data(set_length=72)
dataset = TensorDataset(X, y)

print(f"Dataset imported, size: {len(dataset)}")

Using: mps
Dataset imported, size: 732


/Users/maciejszczesny/SecondaryStructureProject/bin/data_loader.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(aa_encode(seq), dtype=torch.long) for seq in df["seq"]


# Splitting the Data & Batching

In [2]:
traindata, valdata, testdata = torch.utils.data.random_split(dataset, [0.80, 0.10, 0.10])

print(f"Train size: {len(traindata)}, Val size: {len(valdata)}, Test size: {len(testdata)}")

train_loader = DataLoader(traindata, batch_size=32, shuffle=True)
val_loader = DataLoader(valdata, batch_size=32, shuffle=False)
test_loader = DataLoader(testdata, batch_size=32, shuffle=False)

Train size: 586, Val size: 73, Test size: 73


# Import Model, Config & Train
This method is inefficient. I did not realise there was an attention pytorch module... anyways.

I am planning to implement [FLASHattention](https://arxiv.org/abs/2205.14135)

In [3]:
from bin.model import Transformer

src_vocab_size = len(AminoAcids) + 1
tgt_vocab_size = len(SecondaryStructures)
d_model = 64
num_heads = 2
num_layers = 3
d_ff = 4*d_model
max_seq_length = set_length
dropout = 0.1


transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

transformer.eval()

Transformer(
  (encoder_embedding): Linear(in_features=21, out_features=64, bias=True)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-2): 3 x EncoderLayer(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=64, out_features=64, bias=True)
        (W_k): Linear(in_features=64, out_features=64, bias=True)
        (W_v): Linear(in_features=64, out_features=64, bias=True)
        (W_o): Linear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (local_convolution): LocalConv(
        (conv): Sequential(
          (0): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,), groups=64)
          (1): GELU(approximate='none')
          (2): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (norm3): LayerN

In [4]:
from bin.train import train_model

transformer = train_model(
    model=transformer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    tgt_vocab_size=tgt_vocab_size,
    num_epochs=300,
    learning_rate=1e-4,
    save_path="./data/weights.pth"
)

RuntimeError: MPS device does not support linear for non-float inputs

# Test

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim

def test_model(model, test_loader, device, tgt_vocab_size, criterion):
    
    model.eval()

    total_correct = 0
    total_positions = 0
    total_test_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            output = model(X_batch)
        
            tgt_output = y_batch



            loss = criterion(
                output.contiguous().view(-1, tgt_vocab_size),
                tgt_output.contiguous().view(-1)
            )
            total_test_loss += loss.item()
            predicted = output.argmax(dim=-1)

            total_correct += (predicted == y_batch).sum().item()
            total_positions += y_batch.numel()


    avg_test_loss = total_test_loss / len(test_loader)
    
    test_accuracy = total_correct / total_positions * 100

    print(f"Test Loss: {avg_test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.2f}%")

    return avg_test_loss, test_accuracy

criterion = nn.CrossEntropyLoss()

test_loss, test_accuracy = test_model(
    model=transformer,
    test_loader=test_loader,
    device=device,
    tgt_vocab_size=tgt_vocab_size,
    criterion=criterion
)

Test Loss: 0.6798
Test Accuracy: 82.65%


In [52]:
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout).to(device)
model.load_state_dict(torch.load("./data/weights.pth", weights_only=True))

np.random.seed(42)

for i in range(9):
    random_row = np.random.randint(0, len(testdata))
    src_example = testdata[random_row][0].unsqueeze(0).to(device)
    true_labels = testdata[random_row][1].unsqueeze(0).to(device)


    model.eval()
    with torch.no_grad():
        prediction = model(src_example)
        predicted_labels = prediction.argmax(-1)
        accuracy = (predicted_labels == true_labels).float().mean().item()

    predicted_sst8 = ss_decode(predicted_labels.squeeze(0).cpu().tolist())
    true_sst8 = ss_decode(true_labels.squeeze(0).cpu().tolist())

    print(f"Accuracy:  {accuracy:.2%}")
    print(f"Predicted: {predicted_sst8}")
    print(f"True:      {true_sst8}")

Accuracy:  90.28%
Predicted: CCCCCCCSCSSCCCSCCCGGGEEEEEEECSCSSCSSCEEEEEETTTEEEEECTTSHHHHHHHHHHHHHHHTC
True:      CCCSCCCSCSSCCCSCCCGGGEEEEEEECSBTTBSSCBEEEEETTTEEEEECTTSHHHHHHHHHHHHHHHHC
Accuracy:  30.56%
Predicted: CCHHHHHHHHHHHTTSCHHHHHHHHHHHHHHHHHHHHHHHTSSSCCTTHHHHHHHHHHHHHHHHHHHHHCCC
True:      CCCCCCCCCCCCCTTHHHHHHHHHHHHHHHTTTTTCCSSGGGGTTTTTIIIIITSCCCCCCCCCCCCCCCCC
Accuracy:  95.83%
Predicted: CCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCCCCCCCC
True:      CTHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHTTCCCCCCCCC
Accuracy:  90.28%
Predicted: CCCCCCCCSHHHHHHHHHHHHTCSEEHHHHHHHTTCCSGGGTHHHHHHHHHTTTEEECTTTCEEEECCEECC
True:      CCCSSCCSSHHHHHHHHHHHHHCSEEHHHHHHHTTCCSGGGTHHHHHHHHHTTSEEECTTTCEEEECCCCCC
Accuracy:  83.33%
Predicted: CCHHHHHHHHHHHTTSCHHHHHHHHHHHHHHHHHHHHHHHTSSSCCTTHHHHHHHHHHHHHHHHHHHHHCCC
True:      CCCCCHHHHHHHHSSSCSHHHHHHHHHHHHHHHHHHHHHHSCTTCCTTHHHHHHHHHHHHHHHHHHHHHHTC
Accuracy:  83.33%
Predicted: CCCCCCBCSHHHHHHHHHHHHHTTTCCHHHHHHHHTCCHHH